# SDG 7 — Adım 2: EPDK Çok Zamanlı Tarife ve Maliyet Senaryosu

**Amaç:** Mevcut Random Forest tahminlerinden, **gerçek EPDK tarife tablosundaki birim fiyatları** kullanarak fatura bileşenleri ve varsayımsal tüketim kaydırma senaryoları hesaplamak.

**Akademik dürüstlük:** 2010'da Fransa'da ölçülmüş tek evin saatlik tüketimini, 4 Nisan 2026 tarihli Türk mesken tarifesine *karşı-olgusal senaryo* olarak uyguluyoruz. Bu, geçmiş dönemin gerçek faturası, Türkiye'de gerçekleşmiş tasarruf ya da cihaz bazlı optimizasyon **değildir**. Sonraki aşamada çok haneli veri ile geliştireceğiz.

**Kaynaklar:** [EPDK resmi tarife tabloları](https://www.epdk.gov.tr/Detay/Icerik/3-1327/elektrik-faturalarina-esas-tarife-tablolari), [EPDK 2026-04-04 kararının özeti ve rakamları](https://gesatlasi.com.tr/veri/elektrik-tarifeleri), [Enerjisa saat aralıkları](https://www.enerjisa.com.tr/tr/sikca-sorulan-sorular/fatura/tarife-degisikligi-yapmam-durumunda-fatura-tutarim-duser-mi).

**Fiyat birimi:** EPDK tablosundaki **kr/kWh** değerleri 100'e bölünerek **TL/kWh**'ye çevrilmiştir. Mesken AG tek terimli çok zamanlı: gündüz **1,950640**, puant **3,747551**, gece **0,513656**, dağıtım **2,424900** TL/kWh (vergiler hariç, EPDK 4 Nisan 2026 tablosu). Hesaplamada ayrıca varsayılan %5 elektrik tüketim vergisi ve %10 KDV gösterilir; faturanın diğer koşulları değişebilir.

**Önemli sınırlama:** Örnek hanenin tüketimi çok yüksek olabilir ve gerçek hayatta Türkiye'de farklı tedarik/tarife koşulları (SKTT) uygulanabilir. Bu nedenle çıktı gerçek bir konut faturası değildir.


## 1. CSV dosyalarını yükle
Önce `test_predictions.csv` ve `epdk_tariff_2026_04_04.csv` dosyalarını bilgisayarından seç. Notebook, iki dosyanın bulunmasını kontrol edecek.


In [ ]:
from pathlib import Path
try:
    from google.colab import files
    files.upload()
except ImportError:
    print("Colab disinda calisiyor. Iki CSV dosyasini notebook klasorune koyun.")
assert Path("test_predictions.csv").exists(), "test_predictions.csv yuklenmedi"
assert Path("epdk_tariff_2026_04_04.csv").exists(), "EPDK tarife dosyasi yuklenmedi"
print("Girdi dosyalari tamam, siradaki hucreyi calistir.")


## 2. Gerçek tarife ile senaryoları hesapla
İlk durumda hiçbir tüketim kaydırılmaz (%0). Ardından puant tüketiminin varsayımsal olarak %10 / %20 / %30 kaydırılabildiği durumlar analiz edilir. Her gün en fazla 2 kWh ek yük 22.00 ve 23.00 saatlerine aktarılır. Her senaryoda toplam günlük kWh aynıdır.


In [ ]:
"""SDG7, Adim 2: EPDK tarife snapshot + day-ahead cost scenario.

Run: python step2_epdk_cost_analysis.py
Input: test_predictions.csv and epdk_tariff_2026_04_04.csv in the same folder.
Outputs: results/ subfolder.

IMPORTANT: Historical French 2010 consumption is priced with a 2026 Turkish
residential low-voltage 3-time tariff only as a counterfactual illustration;
these are NOT actual bills or experimentally demonstrated savings.  The
adjustable share of total peak-hour load is assumed flexible, not metered by
appliance. The 2026 rates are a dated tariff snapshot, not guaranteed to remain
current.  Taxes are illustrative; the consumer's bill may differ substantially.
"""
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path.cwd()
INPUT = BASE / "test_predictions.csv"
TARIFF = BASE / "epdk_tariff_2026_04_04.csv"
OUTPUT = BASE / "results"
OUTPUT.mkdir(exist_ok=True)

# IMPORTANT: Published EPDK table is in kr/kWh; CSV uses TL/kWh (divide by 100).
# These are EPDK tariffs effective 2026-04-04, not 2010 tariff values.
TAX_ON_ENERGY = 0.05  # illustrative mesken electricity consumption tax
VAT = 0.10           # illustrative mesken VAT
SHIFT_SHARES = (0.0, 0.10, 0.20, 0.30)
NIGHT_HOURS = (22, 23)  # illustrative device-operating window
MAX_EXTRA_KWH_PER_HOUR = 1.0  # explicit user scheduling assumption


def period(hour):
    """Constant UTC+3-adjusted meter clock assumption, verified for modern meters."""
    return "Gunduz" if 6 <= hour < 17 else "Puant" if 17 <= hour < 22 else "Gece"


def bill_components(hour_df, consumption_column, day_rates):
    """Return energy, distribution, tax and total for 24-hour scenario."""
    energy = float((hour_df[consumption_column] * hour_df.period.map(day_rates)).sum())
    dist_rate = float(hour_df.distribution_tl_per_kwh.iloc[0])
    distribution = float(hour_df[consumption_column].sum() * dist_rate)
    electricity_tax = energy * TAX_ON_ENERGY
    vat_amount = (energy + distribution + electricity_tax) * VAT
    return {
        "energy_tl": energy,
        "distribution_tl": distribution,
        "electricity_tax_tl": electricity_tax,
        "vat_tl": vat_amount,
        "estimated_total_tl": energy + distribution + electricity_tax + vat_amount,
    }


def shift_one_day(group, share):
    """Hypothetical shift; no claim of appliance-level identification."""
    df = group.copy()
    df["scheduled_kwh"] = df["forecast_kwh"].to_numpy()
    peak_mask = df.period.eq("Puant")
    peak = float(df.loc[peak_mask, "forecast_kwh"].sum())
    potential = share * peak
    max_shift = len(NIGHT_HOURS) * MAX_EXTRA_KWH_PER_HOUR
    shifted = min(potential, max_shift)
    if peak > 0 and shifted > 0:
        df.loc[peak_mask, "scheduled_kwh"] *= (1 - shifted / peak)
        for h in NIGHT_HOURS:
            df.loc[df.hour.eq(h), "scheduled_kwh"] += shifted / len(NIGHT_HOURS)
    assert np.isclose(df.forecast_kwh.sum(), df.scheduled_kwh.sum())
    assert (df.scheduled_kwh >= -1e-9).all()
    return df, shifted


def main():
    rates = pd.read_csv(TARIFF)
    expected = {"Gunduz", "Puant", "Gece"}
    assert set(rates.period) == expected, "Missing or duplicate tariff periods"
    assert rates.effective_date.nunique() == 1
    assert (rates.energy_tl_per_kwh > 0).all()
    assert (rates.distribution_tl_per_kwh == rates.distribution_tl_per_kwh.iloc[0]).all()
    rate_map = dict(zip(rates.period, rates.energy_tl_per_kwh))
    df = pd.read_csv(INPUT, parse_dates=["datetime"])
    required = {"datetime", "actual_kwh", "random_forest_prediction", "previous_day_prediction", "previous_week_prediction"}
    assert required.issubset(df.columns), f"Missing columns: {required - set(df.columns)}"
    assert df[list(required)].notna().all().all()
    assert not df.datetime.duplicated().any()
    df = df.sort_values("datetime").reset_index(drop=True)
    df["date"] = df.datetime.dt.date
    df["hour"] = df.datetime.dt.hour
    df["period"] = df.hour.map(period)
    df["forecast_kwh"] = df.random_forest_prediction.clip(lower=0)
    df["energy_tl_per_kwh"] = df.period.map(rate_map)
    df["distribution_tl_per_kwh"] = float(rates.distribution_tl_per_kwh.iloc[0])
    completeness = df.groupby("date").hour.agg(["size", "nunique"])
    valid = completeness.index[(completeness["size"] == 24) & (completeness["nunique"] == 24)]
    df = df[df.date.isin(valid)].copy()
    assert len(df) > 0, "No complete day data"

    metrics = {}
    for label, col in [("Previous day", "previous_day_prediction"), ("Previous week", "previous_week_prediction"), ("Random Forest", "random_forest_prediction")]:
        error = df.actual_kwh - df[col]
        metrics[label] = {"mae_kwh": float(error.abs().mean()), "rmse_kwh": float(np.sqrt(np.mean(error**2)))}
    pd.DataFrame(metrics).T.rename_axis("method").to_csv(OUTPUT / "forecast_metrics.csv")

    daily_rows = []
    hourly_rows = []
    for day, g in df.groupby("date", sort=True):
        baseline = bill_components(g, "forecast_kwh", rate_map)
        for share in SHIFT_SHARES:
            moved, shifted_kwh = shift_one_day(g, share)
            compared = bill_components(moved, "scheduled_kwh", rate_map)
            row = {
                "date": str(day), "scenario_share": share,
                "predicted_daily_kwh": float(g.forecast_kwh.sum()),
                "shifted_kwh": shifted_kwh,
                "baseline_cost_tl": baseline["estimated_total_tl"],
                "scheduled_cost_tl": compared["estimated_total_tl"],
                "savings_tl": baseline["estimated_total_tl"] - compared["estimated_total_tl"],
                "baseline_energy_tl": baseline["energy_tl"],
                "scheduled_energy_tl": compared["energy_tl"],
            }
            daily_rows.append(row)
            if share == 0.20:
                temp = moved[["datetime", "date", "hour", "period", "forecast_kwh", "scheduled_kwh", "energy_tl_per_kwh"]].copy()
                hourly_rows.append(temp)
    daily = pd.DataFrame(daily_rows)
    daily["savings_pct"] = np.where(daily.baseline_cost_tl > 0, 100 * daily.savings_tl / daily.baseline_cost_tl, np.nan)
    daily.to_csv(OUTPUT / "daily_tariff_scenarios.csv", index=False)
    hourly = pd.concat(hourly_rows, ignore_index=True)
    hourly.to_csv(OUTPUT / "hourly_example_20percent.csv", index=False)

    agg = daily.groupby("scenario_share").agg(days=("date", "count"),
        consumption_kwh=("predicted_daily_kwh", "sum"),
        shifted_kwh=("shifted_kwh", "sum"),
        baseline_tl=("baseline_cost_tl", "sum"),
        scheduled_tl=("scheduled_cost_tl", "sum"),
        saved_tl=("savings_tl", "sum"))
    agg["saved_pct"] = 100 * agg.saved_tl / agg.baseline_tl
    agg.to_csv(OUTPUT / "scenario_summary.csv")

    first_day = hourly.loc[hourly.date == hourly.date.min()]
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
    ax1.plot(first_day.hour, first_day.forecast_kwh, "o-", label="Forecast: original")
    ax1.plot(first_day.hour, first_day.scheduled_kwh, "s-", label="Hypothetical shift (20%)")
    ax1.set_ylabel("kWh per hour")
    ax1.legend(loc="upper right")
    ax1.grid(alpha=0.2)
    ax2.step(first_day.hour, first_day.energy_tl_per_kwh, where="mid", label="Energy tariff (TL/kWh)")
    ax2.set_ylabel("TL/kWh, energy only")
    ax2.set_xlabel("Hour of day (hypothetical Turkish meter clock)")
    ax2.set_xticks(range(0, 24, 2))
    ax2.grid(alpha=0.2)
    fig.suptitle("EPDK three-time tariff scenario — illustrative; not actual savings")
    fig.tight_layout()
    fig.savefig(OUTPUT / "tariff_forecast_example.png", dpi=160)
    plt.close(fig)

    with open(OUTPUT / "results_notes.txt", "w", encoding="utf-8") as f:
        f.write("SDG7 – EPDK dated rate snapshot, illustrative cost and load shifting\n")
        f.write("EPDK effective date: 2026-04-04; 2026 rates mapped to 2010 French consumption.\n")
        f.write("Modern Turkish meter hour periods: 06-17 day / 17-22 peak / 22-06 night.\n")
        f.write("Energy prices (TL/kWh): " + str(rate_map) + "\n")
        f.write(f"Distribution (TL/kWh): {rates.distribution_tl_per_kwh.iloc[0]:.4f}; tax assumptions: ETV 5%, VAT 10%.\n")
        f.write("Sensitive/regulated pricing cases (SKTT, other suppliers) not supported.\n")
        f.write("SHIFT: fraction of entire predicted peak-hour load assumed flexible; capped to 2 kWh/day and spread equally over 22:00 and 23:00. Not appliance-level measurements.\n")
        f.write("These are 216 selected complete evaluation days, NOT 216 consecutive billing days.\n")
        f.write("Forecast errors are in kWh, cost results are TL, and simulated changes are NOT empirically measured savings.\n\n")
        f.write("FORECAST METRICS\n" + pd.DataFrame(metrics).T.to_string() + "\n\n")
        f.write("SENSITIVITY SCENARIOS\n" + agg.to_string(float_format=lambda x: f"{x:.3f}") + "\n")
    print((OUTPUT / "results_notes.txt").read_text(encoding="utf-8"))

if __name__ == "__main__":
    main()


## 3. Sonuçları tek pakette indir
Notebook `results` klasörüne MAE/RMSE, maliyet-tasarruf tabloları ve örnek grafik kaydeder. Bu hücre hepsini zip olarak indirmeyi dener.


In [ ]:
from zipfile import ZipFile
with ZipFile("EPDK_Sonuclar.zip", "w") as package:
    for item in sorted(Path("results").glob("*")):
        if item.is_file():
            package.write(item, arcname=item.name)
print("Dosya hazır: EPDK_Sonuclar.zip")
try:
    from google.colab import files
    files.download("EPDK_Sonuclar.zip")
except ImportError:
    print("Zip dosyasını notebook klasöründe bulabilirsin.")
